# 06 · Ablación: ¿la ventaja viene del modelo o del tratamiento de la censura?

En `04_inventario`, LightGBM se entrena **sin días censurados** y los métodos clásicos con **ventas** (ceros de agotado incluidos). Así no se puede separar el efecto del modelo del efecto de limpiar la censura.

Aquí los métodos clásicos se reentrenan con la **demanda imputada** (los días de agotado se reemplazan por la media móvil de 28 días previos sin censura, la misma demanda que usa la simulación) y todos pasan por la misma política (R, S) con calibración clase × quintil de volumen.

- **TSB / AutoETS**: entrenados con ventas (como en `04`).
- **TSB_imp / AutoETS_imp**: entrenados con demanda imputada.
- AutoETS se ajusta una vez en el primer origen (13-jul-2015) y después solo actualiza estados (`refit=False`), igual que LightGBM, que se entrena una sola vez. Reajustarlo cada semana tomaría horas.

Las celdas 1–4 repiten el código de `04_inventario` para partir exactamente de la misma base.

In [1]:
from pathlib import Path
from datetime import timedelta
import sys
import time
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import plotly.express as px
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, SeasonalNaive, TSB

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PROC = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
REP = ROOT / "reports"
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))
from features import construir_features, FEATS, CAT_FEATS
from simulate import simular_rs

# --- Supuestos de la política (R, S) ---
R, L = 7, 7               # revisión semanal, lead time de 7 días
PROT = R + L - 1          # días que cubre cada pedido: del día siguiente hasta antes del próximo arribo
N_CALIB, N_SIM = 13, 26   # semanas de calibración y de simulación
H_ANUAL = 0.25            # costo de mantener: 25% anual del precio
MARGEN = 0.30             # costo de faltante: margen perdido por unidad no vendida
WARMUP = 14               # días iniciales que no se miden
ALPHAS = [0.50, 0.70, 0.80, 0.90, 0.95, 0.98, 0.99]

df = pl.read_parquet(PROC / "m5_CA_1.parquet")
seg = pl.read_parquet(PROC / "segmentacion_CA_1.parquet")
mascara = pl.read_parquet(PROC / "mascara_CA_1.parquet")
ids_validos = (pd.read_parquet(PROC / "cv_baselines_CA_1.parquet", columns=["unique_id"])
                 ["unique_id"].unique().tolist())

# --- Calendario (todo antes de la prueba final reservada) ---
dev_end = pd.Timestamp(df["date"].max()) - timedelta(days=28)
last_review = dev_end - timedelta(days=PROT)
reviews_sim = [last_review - timedelta(days=7 * k) for k in range(N_SIM)][::-1]
sim_start = reviews_sim[0]
last_calib = sim_start - timedelta(days=PROT + 1)
origins_calib = [last_calib - timedelta(days=7 * k) for k in range(N_CALIB)][::-1]
train_end = origins_calib[0]
origenes = pd.date_range(origins_calib[0], last_review, freq="7D")
N_WINDOWS = len(origenes)

print("Entrenamiento LightGBM hasta:", train_end.date())
print("Calibración (orígenes):      ", origins_calib[0].date(), "→", last_calib.date(), f"({N_CALIB})")
print("Simulación (revisiones):     ", sim_start.date(), "→", last_review.date(), f"({N_SIM})")
print("Último día simulado:         ", dev_end.date(), f"| orígenes totales: {N_WINDOWS}")

Entrenamiento LightGBM hasta: 2015-07-13
Calibración (orígenes):       2015-07-13 → 2015-10-05 (13)
Simulación (revisiones):      2015-10-19 → 2016-04-11 (26)
Último día simulado:          2016-04-24 | orígenes totales: 40


In [2]:
dev = (df.filter(pl.col("date") <= dev_end.date())
         .with_columns(pl.col("id").cast(pl.Utf8))
         .filter(pl.col("id").is_in(ids_validos)))

inicio = dev.group_by("id").agg(inicio=pl.col("date").min())
fuera = (mascara.filter((pl.col("date") > train_end.date())
                        & pl.col("tipo").is_in(["inactivo", "pre_lanzamiento"]))
                ["id"].unique().to_list())
universo = sorted(
    inicio.filter(pl.col("inicio") <= (train_end - timedelta(days=112)).date())
          .filter(~pl.col("id").is_in(fuera))["id"].to_list()
)
print(f"Productos simulados: {len(universo):,} de {len(ids_validos):,} "
      f"(excluidos por inactividad en el periodo: {len(fuera):,})")

d = (
    dev.filter(pl.col("id").is_in(universo))
    .join(mascara.select(["id", "date", "tipo", "excluir"]), on=["id", "date"], how="left")
    .with_columns(pl.col("excluir").fill_null(False), pl.col("tipo").fill_null("normal"),
                  ventas=pl.col("sales").cast(pl.Float64))
    .sort(["id", "date"])
    .with_columns(limpia=pl.when(pl.col("excluir")).then(pl.lit(None)).otherwise(pl.col("ventas")))
    .with_columns(
        num=pl.col("limpia").fill_null(0).rolling_sum(28).shift(1).over("id"),
        den=(~pl.col("excluir")).cast(pl.Float64).rolling_sum(28).shift(1).over("id"),
        media_prod=pl.col("limpia").mean().over("id"),
    )
    .with_columns(imp=pl.when(pl.col("den") > 0).then(pl.col("num") / pl.col("den"))
                        .otherwise(pl.col("media_prod")))
    .with_columns(demanda=pl.when(pl.col("tipo") == "agotado").then(pl.col("imp"))
                            .otherwise(pl.col("ventas")))
    # Navidad: la tienda cierra, la demanda real es 0
    .with_columns(demanda=pl.when((pl.col("date").dt.month() == 12) & (pl.col("date").dt.day() == 25))
                            .then(0.0).otherwise(pl.col("demanda")))
)

ventana = d.filter(pl.col("date") > train_end.date())
print(f"Días-producto con demanda imputada (agotado) en calibración + simulación: "
      f"{(ventana['tipo'] == 'agotado').mean():.1%}")

Productos simulados: 2,467 de 3,041 (excluidos por inactividad en el periodo: 569)


Días-producto con demanda imputada (agotado) en calibración + simulación: 2.6%


In [3]:
PARAMS = dict(objective="tweedie", tweedie_variance_power=1.1,
              n_estimators=500, learning_rate=0.05, num_leaves=127, min_child_samples=100,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
              n_jobs=-1, verbose=-1)

pdf = construir_features(dev, mascara)
tr = pdf[(pdf["date"] <= train_end) & (~pdf["excluir"]) & (pdf["navidad"] == 0)]

t0 = time.time()
modelo = lgb.LGBMRegressor(**PARAMS)
modelo.fit(tr[FEATS], tr["sales"], categorical_feature=CAT_FEATS)

te = pdf[(pdf["date"] > train_end) & (pdf["id"].isin(universo))].copy()
te["LGB"] = np.clip(modelo.predict(te[FEATS]), 0, None)
P_lgb = te.pivot(index="date", columns="id", values="LGB").reindex(columns=universo)
print(f"Entrenado con {len(tr):,} filas en {time.time() - t0:.0f} s | pronósticos: {P_lgb.shape}")

Entrenado con 3,317,042 filas en 68 s | pronósticos: (286, 2467)


In [4]:
# --- Pronósticos clásicos con ventas y con demanda imputada ---
# n_jobs=1: con varios procesos, statsforecast puede bloquearse junto con el pool de hilos de Polars
def cv_clasicos(columna, modelos, refit):
    Y = d.select(unique_id="id", ds="date", y=pl.col(columna).cast(pl.Float32)).to_pandas()
    Y["ds"] = pd.to_datetime(Y["ds"]).astype("datetime64[ns]")
    cv = (StatsForecast(models=modelos, freq="D", n_jobs=1)
          .cross_validation(df=Y, h=PROT, step_size=7, n_windows=N_WINDOWS, refit=refit))
    if "unique_id" not in cv.columns:
        cv = cv.reset_index()
    cv["cutoff"] = pd.to_datetime(cv["cutoff"]).astype("datetime64[ns]")
    assert cv["cutoff"].min() == origenes[0] and cv["cutoff"].max() == origenes[-1], "Orígenes no coinciden"
    nombres = [c for c in cv.columns if c not in ("unique_id", "ds", "cutoff", "y")]
    cv[nombres] = cv[nombres].clip(lower=0)
    return {m: (cv.groupby(["cutoff", "unique_id"])[m].sum().unstack()
                  .reindex(index=origenes, columns=universo)) for m in nombres}

trabajos = [
    ("ventas",  "",     [SeasonalNaive(season_length=7), TSB(alpha_d=0.1, alpha_p=0.1)], True),
    ("demanda", "_imp", [TSB(alpha_d=0.1, alpha_p=0.1)], True),
    ("ventas",  "",     [AutoETS(season_length=7)], False),
    ("demanda", "_imp", [AutoETS(season_length=7)], False),
]
F_cv = {}
for columna, sufijo, modelos, refit in trabajos:
    t0 = time.time()
    for m, Fm in cv_clasicos(columna, modelos, refit).items():
        assert not Fm.isna().any().any(), f"Huecos en {m}{sufijo}"
        F_cv[m + sufijo] = Fm
    print(f"{columna:8} {[type(x).__name__ for x in modelos]}: {(time.time() - t0) / 60:.1f} min")

ventas   ['SeasonalNaive', 'TSB']: 0.1 min


demanda  ['TSB']: 0.1 min


ventas   ['AutoETS']: 10.0 min


demanda  ['AutoETS']: 10.0 min


In [5]:
fechas = pd.date_range(origenes[0] + timedelta(days=1), dev_end)
pdd = d.select(["id", "date", "demanda", "sell_price"]).to_pandas()
pdd["date"] = pd.to_datetime(pdd["date"]).astype("datetime64[ns]")
D_full = pdd.pivot(index="date", columns="id", values="demanda").reindex(index=fechas, columns=universo)
precio_full = pdd.pivot(index="date", columns="id", values="sell_price").reindex(index=fechas, columns=universo)
P_lgb = P_lgb.reindex(index=fechas)
assert not D_full.isna().any().any(), "Huecos en la matriz de demanda"
assert not P_lgb.isna().any().any(), "Huecos en los pronósticos de LightGBM"

def suma_por_origen(M):
    """Suma de los días o+1 … o+PROT para cada origen o."""
    return pd.DataFrame({o: M.loc[o + timedelta(days=1): o + timedelta(days=PROT)].sum()
                         for o in origenes}).T

A = suma_por_origen(D_full)
F = {"LGB": suma_por_origen(P_lgb)}
F.update(F_cv)

calib = [o for o in origenes if o <= last_calib]
simo = [o for o in origenes if o >= sim_start]
print('Modelos:', list(F))


Modelos: ['LGB', 'SeasonalNaive', 'TSB', 'TSB_imp', 'AutoETS', 'AutoETS_imp']


In [6]:
clase = (seg.select(id=pl.col("id").cast(pl.Utf8), sb_class="sb_class").to_pandas()
            .set_index("id")["sb_class"].reindex(universo))

q = {}
for m, Fm in F.items():
    z = (A.loc[calib] - Fm.loc[calib]) / np.sqrt(Fm.loc[calib] + 1)
    zl = z.T.rename_axis(index="id", columns="origen").stack().rename("z").reset_index()
    zl["clase"] = zl["id"].map(clase)
    q[m] = zl.groupby("clase")["z"].quantile(ALPHAS).unstack()


sim_fechas = pd.date_range(sim_start, dev_end)
D_sim = D_full.loc[sim_fechas].to_numpy()
precio = precio_full.loc[sim_fechas].to_numpy()
revision = sim_fechas.isin(reviews_sim)
idx_rev = np.where(revision)[0]
assert idx_rev[0] == 0 and len(idx_rev) == N_SIM

def metricas(res, cols=slice(None)):
    s = slice(WARMUP, None)
    dem, vend, perd, inv, p = (x[s][:, cols] for x in
                               (D_sim, res["vendido"], res["perdido"], res["inventario"], precio))
    c_mant = (inv * p * H_ANUAL / 365).sum()
    c_falt = (perd * p * MARGEN).sum()
    return {"fill_rate": vend.sum() / dem.sum(),
            "inv_valor_prom": (inv * p).sum(axis=1).mean(),
            "dias_cobertura": inv.sum() / dem.sum(),
            "costo_mantener": c_mant,
            "costo_faltante": c_falt,
            "costo_total": c_mant + c_falt}


N_BINS = 5

def calibrar_por_volumen(Fm_all):
    """Cuantiles de z por (clase, quintil de volumen) con los errores fuera de muestra de la calibración."""
    Fc = Fm_all.loc[calib]
    z = (A.loc[calib] - Fc) / np.sqrt(Fc + 1)
    zl = pd.DataFrame({"id": np.tile(universo, len(calib)),
                       "f": Fc.to_numpy().ravel(), "z": z.to_numpy().ravel()})
    zl["clase"] = zl["id"].map(clase)
    edges = np.unique(np.quantile(zl["f"], np.linspace(0, 1, N_BINS + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    zl["bin"] = pd.cut(zl["f"], edges, labels=False).astype(int)
    return zl.groupby(["clase", "bin"])["z"].quantile(ALPHAS).unstack(), edges


filas = []
for m, Fm_all in F.items():
    tabla_q, edges = calibrar_por_volumen(Fm_all)
    Fm = Fm_all.loc[reviews_sim].to_numpy()
    bins = pd.cut(Fm.ravel(), edges, labels=False).astype(int)
    claves = pd.MultiIndex.from_arrays([np.tile(clase.to_numpy(), len(reviews_sim)), bins])
    for a in ALPHAS:
        qa = tabla_q[a].reindex(claves).to_numpy().reshape(Fm.shape)
        qa = np.where(np.isnan(qa), np.tile(clase.map(q[m][a]).to_numpy(), (len(reviews_sim), 1)), qa)
        S = np.zeros_like(D_sim)
        S[idx_rev] = np.maximum(0.0, Fm + qa * np.sqrt(Fm + 1))
        res = simular_rs(D_sim, S, revision, L, inv_inicial=np.ceil(S[0]))
        filas.append({"modelo": m, "alpha": a, **metricas(res)})

tabla = pd.DataFrame(filas)

# Control: LGB, TSB y SeasonalNaive deben reproducir la simulación publicada de 04_inventario
pub = pd.read_csv(REP / "simulacion_inventario_volumen_CA_1.csv")
chk = pub.merge(tabla, on=["modelo", "alpha"], suffixes=("_04", "_06"))
assert len(chk) == len(pub) and np.allclose(chk["costo_total_04"], chk["costo_total_06"], rtol=1e-6), "No reproduce 04"
print("Control OK: LGB, TSB y SeasonalNaive reproducen 04_inventario")
tabla.to_csv(REP / "ablacion_censura_CA_1.csv", index=False)

Control OK: LGB, TSB y SeasonalNaive reproducen 04_inventario


In [7]:
# --- Resumen ---
calib = [o for o in origenes if o <= last_calib]
simo = [o for o in origenes if o >= sim_start]

def inv_en(modelo, fr):
    c = tabla.query("modelo == @modelo").sort_values("fill_rate")
    if not (c["fill_rate"].min() <= fr <= c["fill_rate"].max()):
        return np.nan
    return np.interp(fr, c["fill_rate"], c["inv_valor_prom"])

opt = tabla.loc[tabla.groupby("modelo")["costo_total"].idxmin()].set_index("modelo")
orden = ["LGB", "AutoETS_imp", "TSB_imp", "AutoETS", "TSB", "SeasonalNaive"]
resumen = pd.DataFrame({
    "Sesgo_sim_%": {m: (F[m].loc[simo] - A.loc[simo]).sum().sum() / A.loc[simo].sum().sum() * 100 for m in orden},
    "Inv_FR95": {m: inv_en(m, 0.95) for m in orden},
    "Alpha_opt": opt["alpha"],
    "Fill_opt": opt["fill_rate"],
    "Costo_opt": opt["costo_total"],
}).loc[orden]
resumen["LGB_vs_%_inv95"] = (resumen.loc["LGB", "Inv_FR95"] / resumen["Inv_FR95"] - 1) * 100
resumen["LGB_vs_%_costo"] = (resumen.loc["LGB", "Costo_opt"] / resumen["Costo_opt"] - 1) * 100
resumen.to_csv(REP / "ablacion_resumen_CA_1.csv")
display(resumen.round({"Sesgo_sim_%": 1, "Inv_FR95": 0, "Fill_opt": 4, "Costo_opt": 0,
                       "LGB_vs_%_inv95": 1, "LGB_vs_%_costo": 1}))

# ¿Qué parte de la brecha LGB vs. TSB cierra la imputación?
for base in ["TSB", "AutoETS"]:
    total = resumen.loc[base, "Costo_opt"] - resumen.loc["LGB", "Costo_opt"]
    cierra = resumen.loc[base, "Costo_opt"] - resumen.loc[f"{base}_imp", "Costo_opt"]
    print(f"{base}: la imputación cierra {cierra / total:.0%} de la brecha de costo con LightGBM")

fig = px.line(tabla[tabla["modelo"].isin(orden)], x="inv_valor_prom", y="fill_rate", color="modelo",
              markers=True, hover_data=["alpha", "costo_total"], category_orders={"modelo": orden},
              labels={"inv_valor_prom": "Inventario promedio (USD)", "fill_rate": "Fill rate", "modelo": "Pronóstico"},
              title="Ablación — métodos clásicos con ventas vs. con demanda imputada (CA_1)")
fig.update_layout(width=1000, height=600, yaxis_tickformat=".0%", xaxis_range=[0, 560_000])
fig.write_html(FIG_DIR / "10_ablacion_censura.html", include_plotlyjs="cdn")
try:
    fig.write_image(FIG_DIR / "10_ablacion_censura.png", scale=2)
except Exception as e:
    print(f"PNG no generado ({e})")
fig.show()

,Sesgo_sim_%,Inv_FR95,Alpha_opt,Fill_opt,Costo_opt,LGB_vs_%_inv95,LGB_vs_%_costo
LGB,0.4,120675.0,0.98,0.9900,31977.0,0.0,0.0
AutoETS_imp,0.7,121055.0,0.98,0.9893,31169.0,-0.3,2.6
TSB_imp,2.5,122834.0,0.98,0.9900,31333.0,-1.8,2.1
AutoETS,-4.1,158963.0,0.95,0.9649,41389.0,-24.1,-22.7
TSB,-2.5,161137.0,0.95,0.9685,43273.0,-25.1,-26.1
SeasonalNaive,-4.6,202179.0,0.90,0.9549,49995.0,-40.3,-36.0


TSB: la imputación cierra 106% de la brecha de costo con LightGBM
AutoETS: la imputación cierra 109% de la brecha de costo con LightGBM


## Conclusiones

| Pronóstico | Sesgo (simulación) | Inventario a fill rate 95% | Costo óptimo |
|---|---|---|---|
| LightGBM | +0.4% | $120,675 | $31,977 |
| AutoETS con demanda imputada | +0.7% | $121,055 | **$31,169** |
| TSB con demanda imputada | +2.5% | $122,834 | $31,333 |
| AutoETS con ventas | −4.1% | $158,963 | $41,389 |
| TSB con ventas | −2.5% | $161,137 | $43,273 |
| Seasonal naive | −4.6% | $202,179 | $49,995 |

- **La ventaja viene del tratamiento de la censura, no del modelo.** Entrenados con demanda imputada, TSB y AutoETS quedan prácticamente empatados con LightGBM: ±2% de inventario a fill rate de 95% y un costo óptimo 2–3% *menor*. La imputación cierra toda la brecha de costo (106% con TSB, 109% con AutoETS).
- El −25% de inventario y el −26% de costo de `04_inventario` miden, en realidad, **ventas censuradas vs. demanda corregida**.
- LightGBM no aporta ventaja en inventario sobre un método clásico bien alimentado; su valor está en que no necesita imputar (aprende directamente sin los días censurados) y en una precisión ligeramente mejor en la prueba final.

**Advertencia:** la demanda imputada se usa para entrenar los modelos `_imp` y también como demanda real en la simulación (2.6% de los días-producto). Eso puede favorecer ligeramente a los modelos que pronostican cerca de la media reciente, incluido LightGBM.